In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark = SparkSession.builder.appName("E2E_Pipeline").getOrCreate()

data = [
    (1, "C001", "Laptop", "50000", "2024-01-01"),
    (2, "C002", "Mobile", None, "2024-01-02"),
    (3, "C003", "Tablet", "20000", "2024-01-03"),
    (4, "C004", "Laptop", "55000", "2024-01-04"),
    (5, "C005", "Headphones", None, "2024-01-05"),
    (6, "C006", "Camera", "30000", "2024-01-06"),
    (7, "C007", "Mobile", "18000", "2024-01-07"),
    (8, "C008", "Watch", "8000", "2024-01-07")
]

columns = ["order_id", "customer_id", "product", "amount", "updated_date"]

df = spark.createDataFrame(data, columns)
df.display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,null,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,null,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
df=df.fillna({"amount":"0"})
df.display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,0,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,0,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
new_row = [(6, "C006", "Camera", 30000, "2024-01-10")]
new_df = spark.createDataFrame(new_row, df.columns)

In [0]:
df=df.withColumn("amount",
                 col("amount").cast("int"))
df=df.withColumn("amount",
                 when(col("amount")==0,lit(1000)).otherwise(col("amount")))
df.display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,1000,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,1000,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
df=df.withColumn("updated_date",
                 to_date("updated_date","yyyy-mm-dd"))
df.display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,1000,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,1000,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
df=df.withColumn("bonus",
                 col("amount")*0.1)
df.display()

order_id,customer_id,product,amount,updated_date,bonus
1,C001,Laptop,50000,2024-01-01,5000.0
2,C002,Mobile,1000,2024-01-02,100.0
3,C003,Tablet,20000,2024-01-03,2000.0
4,C004,Laptop,55000,2024-01-04,5500.0
5,C005,Headphones,1000,2024-01-05,100.0
6,C006,Camera,30000,2024-01-06,3000.0
7,C007,Mobile,18000,2024-01-07,1800.0
8,C008,Watch,8000,2024-01-07,800.0


In [0]:
df=df.withColumn("category",
                 when(col("amount")>=20000,"high").otherwise("low"))
df.display()

order_id,customer_id,product,amount,updated_date,bonus,category
1,C001,Laptop,50000,2024-01-01,5000.0,high
2,C002,Mobile,1000,2024-01-02,100.0,low
3,C003,Tablet,20000,2024-01-03,2000.0,high
4,C004,Laptop,55000,2024-01-04,5500.0,high
5,C005,Headphones,1000,2024-01-05,100.0,low
6,C006,Camera,30000,2024-01-06,3000.0,high
7,C007,Mobile,18000,2024-01-07,1800.0,low
8,C008,Watch,8000,2024-01-07,800.0,low


In [0]:
from pyspark.sql.types import *
def amount_bucket(amount):
    if amount<=10000:
        return "low"
    elif amount>10000 and amount<=30000:
        return "medium"
    else:
        return "high"
my_udf=udf(amount_bucket,StringType())
udf_df=df.withColumn("amount_bucket",my_udf(col("amount")))
udf_df.display()

order_id,customer_id,product,amount,updated_date,bonus,category,amount_bucket
1,C001,Laptop,50000,2024-01-01,5000.0,high,high
2,C002,Mobile,1000,2024-01-02,100.0,low,low
3,C003,Tablet,20000,2024-01-03,2000.0,high,medium
4,C004,Laptop,55000,2024-01-04,5500.0,high,high
5,C005,Headphones,1000,2024-01-05,100.0,low,low
6,C006,Camera,30000,2024-01-06,3000.0,high,medium
7,C007,Mobile,18000,2024-01-07,1800.0,low,medium
8,C008,Watch,8000,2024-01-07,800.0,low,low


In [0]:
df.write.mode("overwrite").parquet("/Volumes/workspace/default/practice")

In [0]:
max_date=df.select(max("updated_date")).collect()[0][0]
print(max_date)

2024-01-07


In [0]:
incremental_df = new_df.filter(col("updated_date") > max_date)
incremental_df.display()

order_id,customer_id,product,amount,updated_date
6,C006,Camera,30000,2024-01-10


In [0]:
final_df=df.dropDuplicates()
final_df.display()

order_id,customer_id,product,amount,updated_date,bonus,category
1,C001,Laptop,50000,2024-01-01,5000.0,high
2,C002,Mobile,1000,2024-01-02,100.0,low
3,C003,Tablet,20000,2024-01-03,2000.0,high
4,C004,Laptop,55000,2024-01-04,5500.0,high
5,C005,Headphones,1000,2024-01-05,100.0,low
6,C006,Camera,30000,2024-01-06,3000.0,high
7,C007,Mobile,18000,2024-01-07,1800.0,low
8,C008,Watch,8000,2024-01-07,800.0,low


In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/practice"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/practice/_SUCCESS,_SUCCESS,0,1776148922000
dbfs:/Volumes/workspace/default/practice/_committed_1976077795126602613,_committed_1976077795126602613,1623,1776148922000
dbfs:/Volumes/workspace/default/practice/_committed_2653932169897162869,_committed_2653932169897162869,524,1776139338000
dbfs:/Volumes/workspace/default/practice/_committed_3902316596362851771,_committed_3902316596362851771,1615,1776148048000
dbfs:/Volumes/workspace/default/practice/_committed_7488264886592558223,_committed_7488264886592558223,1334,1776147546000
dbfs:/Volumes/workspace/default/practice/_committed_904513624069889818,_committed_904513624069889818,1615,1776147966000
dbfs:/Volumes/workspace/default/practice/_committed_vacuum3243689271713034310,_committed_vacuum3243689271713034310,96,1776147547000
dbfs:/Volumes/workspace/default/practice/_started_1976077795126602613,_started_1976077795126602613,0,1776148921000
dbfs:/Volumes/workspace/default/practice/_started_3902316596362851771,_started_3902316596362851771,0,1776148048000
dbfs:/Volumes/workspace/default/practice/_started_7488264886592558223,_started_7488264886592558223,0,1776147545000


In [0]:
dbutils.widgets.text("input_path", "/Volumes/workspace/default/practice/source")
dbutils.widgets.text("output_path", "/Volumes/workspace/default/practice/target")
print(dbutils.widgets.get("input_path"))
print(dbutils.widgets.get("output_path"))

/Volumes/workspace/default/practice/source
/Volumes/workspace/default/practice/target


In [0]:
dbutils.widgets.text("last_date", "2024-01-01")
dbutils.widgets.dropdown("mode", "full", ["full", "incremental"])
last_date = dbutils.widgets.get("last_date")
mode = dbutils.widgets.get("mode")
print(last_date)
print(mode)

2024-01-01
incremental
